In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent 
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

import torch
import torchaudio
from IPython.display import Audio, display
import librosa

# Import our processor
from processors.wrapper_encodec import EncodecProcessor

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Multiscale FFT Loss
from auxiliar.losses import MultiscaleSpectrogramLoss
mss_loss = MultiscaleSpectrogramLoss(scales=[8192, 4096, 2048, 1024], 
                                    overlap=0.75)
spectral_loss_fn = lambda x,y: mss_loss(x.view(1,-1), y.view(1,-1))

# Initialize 48kHz processor
sr = 48000
processor_48k_streamable = EncodecProcessor(sr=sr, streamable=True, )
processor_48k_chunks     = EncodecProcessor(sr=sr, streamable=False)

def full_autoencoding(audio_tensor, processor):
    sr = processor.sample_rate
    latent_cont_list, latent_cont_meta = processor.audio_to_latents(audio_tensor, sr)
    latent_cont_list = torch.cat(latent_cont_list, dim=-1)
    audio_from_latent_cont_48k = processor.decode_latents_audio(latent_cont_list, metadata=latent_cont_meta)
    return audio_from_latent_cont_48k    

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ All imports successful
PyTorch version: 2.10.0+cu128
Torchaudio version: 2.10.0+cu128
CUDA available: True
Initializing EnCodec processor on device: cuda
Streamable mode: True
  → Enabling streamable mode (disabling chunking)
✓ EnCodec 48kHz model loaded (HuggingFace) - streamable
✓ Sample rate: 48000 Hz
✓ Frame rate: 150 Hz
Initializing EnCodec processor on device: cuda
Streamable mode: False
  → Enabling chunked mode with chunk_length_s=1.0s and overlap=1.0%
✓ EnCodec 48kHz model loaded (HuggingFace) - chunked (1s chunks, 1% overlap)
✓ Sample rate: 48000 Hz
✓ Frame rate: 150 Hz


In [ ]:
def experiment(audio_path, length_seconds=10, overlap_time=0.02, segment_time=0.14):
    # Load audio
    audio_input, _ = librosa.load(audio_path, sr=sr, mono=True)
    audio_input = torch.tensor(audio_input).unsqueeze(0)  # Add batch dimension
    audio_input = audio_input[:, :int(sr*length_seconds)].to(processor_48k_streamable.device) 
    # print(f"\nOriginal audio shape: {audio_input.shape}")

    audio_full_autoencoded_stream = full_autoencoding(audio_input, processor_48k_streamable)
    audio_full_autoencoded_chunks = full_autoencoding(audio_input, processor_48k_chunks)
    # print(f"\nFull autoencoded audio shape: {audio_full_autoencoded.shape}")

    overlap_samples = int(sr * overlap_time)
    segment_samples = int(sr * segment_time)
    hop_time        = segment_time - overlap_time
    hop_samples     = segment_samples - overlap_samples

    # Segmentation
    segments = []
    for start in range(0, audio_input.shape[1], hop_samples):
        end = start + segment_samples
        segment = audio_input[:, start:end]
        # Pad if last segment is shorter than segment_samples
        if segment.shape[1] < segment_samples:
            padding = segment_samples - segment.shape[1]
            segment = torch.nn.functional.pad(segment, (0, padding))
        segments.append(segment)

    # encoding each segment separately
    latents   = []
    metadatas = []
    for i in range(len(segments)):
        latent, meta = processor_48k_streamable.audio_to_latents(segments[i], sr)
        latents.append(latent)
        metadatas.append(meta)

    # Decoding each segment separately
    reconstructed_segments = []
    for i in range(len(latents)):
        reconstructed = processor_48k_streamable.decode_latents_audio(torch.cat(latents[i], dim=-1), metadata=metadatas[i])
        reconstructed_segments.append(reconstructed)

    # Window function = left part of hann + ones + right part of hann
    window = torch.ones(segment_samples-2*overlap_samples, device=audio_input.device)
    hann_window = torch.hann_window(overlap_samples * 2, device=audio_input.device)
    left_hann = hann_window[:overlap_samples]
    right_hann = hann_window[overlap_samples:]
    window = torch.cat([left_hann, window, right_hann])

    # Overlap and add
    reconstructed_audio = torch.zeros((audio_input.shape[0], 2, audio_input.shape[-1]+segment_samples), device=audio_input.device)
    for i in range(len(reconstructed_segments)):
        start = i * hop_samples
        end = start + segment_samples
        reconstructed_audio[:, :, start:end] += reconstructed_segments[i][:, :segment_samples] * window
    reconstructed_audio = reconstructed_audio[:, :, :audio_input.shape[1]]  # Trim to original length
    # print(f"\nReconstructed audio shape: {reconstructed_audio.shape}")

    # Making all of them the same shape for loss computation and listening
    audio_input                   = audio_input[0]
    reconstructed_audio           = reconstructed_audio[0].mean(dim=0, keepdim=False)[:audio_input.shape[0]]
    audio_full_autoencoded_stream = audio_full_autoencoded_stream[0].mean(dim=0, keepdim=False)[:audio_input.shape[0]]
    audio_full_autoencoded_chunks = audio_full_autoencoded_chunks[0].mean(dim=0, keepdim=False)[:audio_input.shape[0]]

    # loss computation
    loss_full_autoencoding_stream = spectral_loss_fn(audio_input, audio_full_autoencoded_stream)
    loss_full_autoencoding_chunks = spectral_loss_fn(audio_input, audio_full_autoencoded_chunks)
    loss_segmented_autoencoding = spectral_loss_fn(audio_input, reconstructed_audio)

    # play!
    print(f"\nOriginal audio")
    display(Audio(audio_input.detach().cpu().numpy(), rate=sr))

    print(f"\n(loss = {loss_full_autoencoding_stream.item():.4f}) Full autoencoded audio from streamable encodec (all at once)")
    display(Audio(audio_full_autoencoded_stream.detach().cpu().numpy(), rate=sr))

    print(f"\n(loss = {loss_full_autoencoding_chunks.item():.4f}) Full autoencoded audio non-streamable encodec (using default chunks processing of 1 second with overlap of 10ms)")
    display(Audio(audio_full_autoencoded_chunks.detach().cpu().numpy(), rate=sr))
    
    print(f"\n(loss = {loss_segmented_autoencoding.item():.4f}) Full autoencoded audio from custom segmented processing with streamable encodec (using segment time of {segment_time} and overlap of {overlap_time})")
    display(Audio(reconstructed_audio.detach().cpu().numpy(), rate=sr))

In [9]:
experiment(audio_path="../samples/guitar.wav", length_seconds=10)


Original audio



(loss = 1.3282) Full autoencoded audio from streamable encodec (all at once)



(loss = 1.2054) Full autoencoded audio non-streamable encodec (using default chunks processing)



(loss = 1.1247) Full autoencoded audio from custom segmented processing with streamable encodec
